In [14]:
# ======================================================
# IMPORT LIBRARY
# ======================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
# ======================================================
# LOAD DATA
# ======================================================

data_path = Path("data") / "matches.xlsx"
df = pd.read_excel(data_path)

print("Shape data awal:", df.shape)

Shape data awal: (19847, 88)


In [20]:
# ======================================================
# FILTER FOOTBALL
# ======================================================

df = df[df["match_main_genre"].astype(str).str.lower() == "football"]

print("Shape setelah filter Football:", df.shape)

Shape setelah filter Football: (6107, 88)


In [21]:
# ======================================================
# PILIH KOLOM
# ======================================================

cols = [
    'match_date_start',
    'match_duration',
    'match_tournament',
    'match_premier_status',
    'match_age_rating',
    'match_content_type',
    'match_coverage',
    'match_genre',
    'match_main_genre',
    'match_channel',
    'match_gender',
    'match_organization',
    'team_home',
    'team_away',
    'match_priority_level'
]

df = df[cols].copy()

In [22]:
# ======================================================
# FEATURE ENGINEERING
# ======================================================

df["match_date_start"] = pd.to_datetime(df["match_date_start"])

df["match_hour"] = df["match_date_start"].dt.hour
df["match_day"] = df["match_date_start"].dt.dayofweek
df["match_month"] = df["match_date_start"].dt.month

df["is_weekend"] = df["match_day"].isin([5,6]).astype(int)
df["is_prime_time"] = df["match_hour"].between(18,23).astype(int)

df.drop(columns=["match_date_start"], inplace=True)

In [23]:
# ======================================================
# CLEAN TARGET
# ======================================================

df["match_priority_level"] = (
    df["match_priority_level"]
    .astype(str)
    .str.strip()
    .str.lower()
)

priority_map = {
    "low":0,
    "medium":1,
    "high":2
}

df["match_priority_level"] = df["match_priority_level"].map(priority_map)

df = df.dropna(subset=["match_priority_level"])

print("\nDistribusi target:")
print(df["match_priority_level"].value_counts())



Distribusi target:
match_priority_level
1.0    5335
2.0     468
0.0     289
Name: count, dtype: int64


In [24]:
# ======================================================
# ENCODING CATEGORICAL FEATURE
# ======================================================

cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:

    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))


In [25]:
# ======================================================
# SPLIT FEATURE & TARGET
# ======================================================

X = df.drop(columns=["match_priority_level"])
y = df["match_priority_level"]

In [26]:
# ======================================================
# TRAIN TEST SPLIT
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (4873, 18)
Test shape: (1219, 18)


In [28]:
# ======================================================
# HITUNG CLASS WEIGHT
# ======================================================

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print("\nClass Weights:")
print(class_weights)



Class Weights:
{0.0: 7.031746031746032, 1.0: 0.3805841924398625, 2.0: 4.3431372549019605}


In [29]:
# ======================================================
# RANDOM FOREST MODEL
# ======================================================

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("\n===== RANDOM FOREST =====")

print("Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))



===== RANDOM FOREST =====
Accuracy: 0.9081214109926169
              precision    recall  f1-score   support

         0.0       0.69      0.86      0.77        58
         1.0       0.97      0.92      0.95      1067
         2.0       0.54      0.78      0.63        94

    accuracy                           0.91      1219
   macro avg       0.73      0.85      0.78      1219
weighted avg       0.93      0.91      0.91      1219



In [30]:
# ======================================================
# XGBOOST MODEL
# ======================================================

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("\n===== XGBOOST =====")

print("Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))



===== XGBOOST =====
Accuracy: 0.9294503691550451
              precision    recall  f1-score   support

         0.0       0.72      0.81      0.76        58
         1.0       0.95      0.97      0.96      1067
         2.0       0.76      0.57      0.65        94

    accuracy                           0.93      1219
   macro avg       0.81      0.78      0.79      1219
weighted avg       0.93      0.93      0.93      1219

